# MEERA API Backend Manual Testing Notebook

This notebook contains cells to make HTTP requests using the `requests` library to test all endpoints of the **MEERA API Backend**.

### Prerequisite:
Make sure your FastAPI server is running. You can start it locally using:
```bash
uv run uvicorn src.main:app --reload
```

In [ ]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"
print(f"Target API Base URL: {BASE_URL}")

## 1. Health Check

Check if the server is up and healthy.

In [ ]:
response = requests.get(f"{BASE_URL}/")
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 2. Department Mapping Master

Upload a department mapping CSV/Excel parsed json payload and fetch it.

In [ ]:
# Upload Master file payload
upload_payload = {
    "records": [
        {
            "office": "Revenue Central Office",
            "division_section": "Property Taxes",
            "sub_section": "Zone A",
            "user": "revenue_officer_1"
        },
        {
            "office": "Revenue Central Office",
            "division_section": "Commercial Taxes",
            "sub_section": "Zone B",
            "user": "revenue_officer_2"
        }
    ],
    "department": "revenue"
}

response = requests.post(f"{BASE_URL}/api/v1/upload/department-mapping-master", json=upload_payload)
print(f"Upload Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Fetch Master data for the department
response = requests.get(f"{BASE_URL}/api/v1/department-mapping-master", params={"department": "revenue"})
print(f"Get Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 3. RTI Queries List & Filtering

Retrieve all RTI query records, optionally filtered by status, assignment parameters, or pagination.

In [ ]:
# List all queries (initially empty if database was clean, or populated from database startup)
response = requests.get(f"{BASE_URL}/api/v1/rti-queries")
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Fetch using validation constraints: Combined filters (should return 400 Bad Request)
response = requests.get(f"{BASE_URL}/api/v1/rti-queries", params={"unassigned_only": "true", "assigned_to": "officer_1"})
print(f"Status Code (Expected 400): {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 4. RTI Query Aggregate Counts

Returns global aggregate counts grouped by status.

In [ ]:
response = requests.get(f"{BASE_URL}/api/v1/rti-queries/count")
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 5. RTI Query Detail & Correspondence

Fetches detailed view of a single query including supporting documents and office notes (sorted newest first).

In [ ]:
# Test Detail lookup for a non-existent ID (should return 404)
response = requests.get(f"{BASE_URL}/api/v1/rti-queries/not_found")
print(f"Status Code (Expected 404): {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 6. Chat & FAQ Deflection Interaction

Submit citizen/officer queries against an RTI query, fetch assistant suggestions, and download the conversation log.

In [ ]:
# POST a user query
chat_payload = {
    "user_query": "Can we request an extension for zone property audit?",
    "user_id": "officer_1",
    "rti_query_id": "query_123"  # Make sure this ID is populated in your DB or matches
}

response = requests.post(f"{BASE_URL}/api/v1/user_query", json=chat_payload)
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

In [ ]:
# POST to request assistant suggestion
sug_payload = {
    "rti_query_id": "query_123",
    "user_id": "officer_1"
}
response = requests.post(f"{BASE_URL}/api/v1/get-suggestion", json=sug_payload)
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

In [ ]:
# GET current chat session and suggestions
response = requests.get(
    f"{BASE_URL}/api/v1/get-session",
    params={"rti_query_id": "query_123", "user_id": "officer_1"}
)
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))